## Parameters

In [ ]:
import anndata as ad
import dotenv
import os

dotenv.load_dotenv()

pert_col = "target_gene"
control = "non-targeting"
adata = ad.read_h5ad(os.getenv("DATA_PATH"), backed=True) # FIXME

## Calculate the mean expression

In [ ]:
from scipy.sparse import csr_matrix
import numpy as np


def get_grouped_mean_var(adata: ad.AnnData, group_column: str="target_gene") -> tuple[np.ndarray, np.ndarray]:
    """Get mean and variance of each group"""
    X = adata.X
    labels = np.asarray(adata.obs[group_column])
    unique_labels, inverse = np.unique(labels, return_inverse=True)
    n_groups = len(unique_labels)
    group_sizes = np.bincount(inverse)

    # Mean
    one_hot = csr_matrix(
        (np.ones_like(inverse), (inverse, np.arange(len(inverse)))),
        shape=(n_groups, X.shape[0])
    )
    group_sums = one_hot @ X  # (n_groups, n_genes)
    group_means = group_sums.toarray() / group_sizes[:, None]

    return group_means, unique_labels


def simulate_with_poisson(means: np.ndarray, sizes: np.ndarray, seed: int=2409) -> tuple[np.ndarray, np.ndarray]:
    """Simulate with Poisson distribution"""
    rng = np.random.default_rng(seed)
    n_groups, n_genes = means.shape

    # Pre-allocate
    total_cells = int(np.sum(sizes))
    X_sim = np.empty((total_cells, n_genes), dtype=np.float32)
    labels_sim = np.empty(total_cells, dtype=int)

    start = 0
    for g in range(n_groups):
        n = sizes[g]
        stop = start + n
        mu = means[g]
        X_sim[start:stop] = rng.poisson(mu, size=(n, n_genes))
        labels_sim[start:stop] = g
        start = stop
    return X_sim, labels_sim


perts = adata.obs[pert_col].unique()
perts = perts[perts != control]
train_adata = adata[adata.obs[pert_col].isin(perts)].to_memory()
means, eval_perts = get_grouped_mean_var(train_adata) # Get lambda
means

## Simulate with Poisson distribution

In [ ]:
def simulate_with_poisson(means: np.ndarray, sizes: np.ndarray, seed: int=2409) -> tuple[np.ndarray, np.ndarray]:
    """Simulate with Poisson distribution"""
    rng = np.random.default_rng(seed)
    n_groups, n_genes = means.shape

    # Pre-allocate
    total_cells = int(np.sum(sizes))
    X_sim = np.empty((total_cells, n_genes), dtype=np.float32)
    labels_sim = np.empty(total_cells, dtype=int)

    start = 0
    for g in range(n_groups):
        n = sizes[g]
        stop = start + n
        mu = means[g]
        X_sim[start:stop] = rng.poisson(mu, size=(n, n_genes))
        labels_sim[start:stop] = g
        start = stop
    return X_sim, labels_sim

size_map = adata.obs[pert_col].value_counts().to_dict()
sizes = [size_map[p] for p in eval_perts]
X_sim, labels_sim = simulate_with_poisson(means, sizes) # Poisson
X_sim.shape

## Evaluation with DES, MAE, and PDS

In [ ]:
from cell_eval import MetricsEvaluator
import pandas as pd

# Turn of functools exception
import warnings
warnings.filterwarnings("ignore", module="functools")

sim = ad.AnnData(
    X_sim,
    obs=pd.DataFrame({pert_col: eval_perts[labels_sim]}),
    var=adata.var,
)
ctrl = adata[adata.obs[pert_col] == control].to_memory()
eval = MetricsEvaluator(
    adata_real=adata.to_memory(),
    adata_pred=ad.concat([sim, ctrl]),
    pert_col=pert_col,
    control_pert=control,
)
results, agg_results = eval.compute(profile="vcc")
agg_results